# LSTM bidireccional para puntuación y capitalización (TP II)

Modelo **(c)** del enunciado: una RNN bidireccional que lee la instancia palabra por palabra **en los dos sentidos** y, en cada palabra, predice las mismas tres etiquetas que la LSTM de `lstm_puntuacion.ipynb`:

| Cabeza | Clases |
|---|---|
| `punt_inicial` | 0 = nada, 1 = `¿` |
| `punt_final` | 0 = nada, 1 = `,`, 2 = `.`, 3 = `?` |
| `capitalizacion` | 0 = minúscula, 1 = Inicial, 2 = mixta, 3 = TODO MAYÚSCULA |

## Qué cambia respecto de la LSTM (y qué no)

* **Cambia solo el modelo.** La palabra `t` ahora ve las anteriores y también las siguientes. Para decidir si hay coma, punto o `?` después de una palabra ayuda mucho saber qué viene después ("qué", "pero", "y").
* **No cambia nada más:** datos, split por instancia, pesos de clase, pérdida, métrica (F1 macro por cabeza), checkpoint de la mejor época y protocolo de búsqueda de hiperparámetros. Todo eso se **importa de `src/busqueda_lstm.py`** (Kenneth), así que la comparación LSTM vs bidireccional es justa: la única diferencia es la arquitectura.

## Cómo se deciden los hiperparámetros

El mismo procedimiento que la LSTM, por etapas y reanudable (si se corta, se retoma desde `salidas/bilstm/`):

| Etapa | Qué prueba |
|---|---|
| A | capas `L` y tamaño oculto `H` **por dirección** |
| B | `dropout`, `lr` y `weight_decay` (un eje a la vez) |
| C | `potencia` de los pesos de clase |
| D | umbral por clase sobre las probabilidades (chequeado en mitades de validación no vistas) |
| final | reentrena la mejor config con 3 semillas y mide en **test una sola vez** |

Reglas para no perseguir el ruido: se elige por score en **validación** (promedio de los 3 F1 macro); un valor nuevo reemplaza al actual solo si lo supera por más de `margen` = 0,005; el test no se toca hasta la etapa final; y se reporta media y desvío entre semillas.

La búsqueda es **más corta** que la de la LSTM porque ya sabemos tres cosas de ella: `L` y `H` casi no importaban (meseta), `potencia` = 0,5 era lo mejor y `lr` = 3e-3 le ganó a 1e-3. Se parte de ahí.

**Archivos que necesita:** `embeddings_palabras.npy` y `etiquetas_palabras.csv` del mismo run (los genera `generar_embeddings.ipynb`) y `busqueda_lstm.py` + `rutas.py`. En Colab, todo en la carpeta de Drive `MyDrive/TP_AA`.

In [ ]:
import argparse
import json
import os
import sys
import time
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
ARCHIVOS_TP = ("busqueda_lstm.py", "rutas.py", "embeddings_palabras.npy", "etiquetas_palabras.csv")


def buscar_carpeta():
    """Carpeta TP_AA ya disponible en el servidor (Drive montado por el comando de la extensión o por otra vía)."""
    for p in (Path("/content/drive/MyDrive/TP_AA"), Path("/content/gdrive/MyDrive/TP_AA"), Path("/content/TP_AA")):
        if p.exists() and all((p / a).exists() for a in ARCHIVOS_TP):
            return p
    for patron in ("*/TP_AA", "*/*/TP_AA", "*/*/*/TP_AA"):   # montado en otro lado
        for raiz in (Path("/content"), Path("/mnt")):
            for p in raiz.glob(patron):
                if p.is_dir() and all((p / a).exists() for a in ARCHIVOS_TP):
                    return p
    return None


def diagnostico():
    """Qué hay en el servidor, para ver si Drive llegó a montarse y dónde."""
    for d in (Path("/content"), Path("/content/drive"), Path("/content/gdrive")):
        if d.exists():
            print(f"contenido de {d}:", sorted(x.name for x in d.iterdir())[:30])


if EN_COLAB:
    CARPETA = buscar_carpeta()   # primero, lo que ya esté montado
    if CARPETA is None:          # si no, intenta montar Drive (en el navegador pide autorizar; en VS Code falla y sigue)
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except Exception as e:
            print("drive.mount() no funcionó:", type(e).__name__)
        CARPETA = buscar_carpeta()
    if CARPETA is None:
        diagnostico()
        raise RuntimeError(
            "No encuentro la carpeta TP_AA con los 4 archivos. Drive no está montado o la carpeta tiene otro nombre. "
            "En VS Code, drive.mount() y auth.authenticate_user() NO funcionan (la extensión rechaza los pedidos de "
            "autorización hechos desde Python): usar Ctrl+Shift+P > 'Colab: Mount Google Drive to Server...', correr la "
            "celda que agrega y volver a correr esta. Si eso tampoco anda, abrir el notebook en "
            "colab.research.google.com, donde drive.mount() funciona directo."
        )
    print("carpeta de trabajo:", CARPETA)
    os.chdir(CARPETA)
    sys.path.insert(0, str(CARPETA))
else:
    RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
    sys.path.insert(0, str(RAIZ / "src"))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

import busqueda_lstm as bl   # datos, pérdida, entrenamiento y etapas de la búsqueda de la LSTM

if os.environ.get("TP_SALIDAS"):   # solo para probar con otros datos
    bl.SALIDAS = Path(os.environ["TP_SALIDAS"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
np.random.seed(0)
print("PyTorch", torch.__version__, "| device:", device, "| salidas:", bl.SALIDAS)

In [ ]:
!nvidia-smi


---
# Paso 1: el modelo

```
 palabra:          w1        w2        w3   ...   wT
 hacia adelante:   h→1 ───►  h→2 ───►  h→3 ──►... h→T       lee de izquierda a derecha
 hacia atrás:      h←1 ◄───  h←2 ◄───  h←3 ◄──... h←T       lee de derecha a izquierda
                    │         │         │          │
 se concatenan:  [h→1;h←1] [h→2;h←2] [h→3;h←3] ... [h→T;h←T]      out (B, T, 2H)
                    │         │         │          │
                  3 cabezas Linear(2H, K) aplicadas a cada palabra
```

* Cada capa tiene **dos** LSTM (una por sentido) con sus propios pesos. La capa 2 recibe como entrada la concatenación de las dos salidas de la capa 1 (`2H` en vez de `H`), y las cabezas también (`2H`).
* `H` es el tamaño oculto **por dirección**: con `H = 64` la salida mide 128, y el modelo tiene aproximadamente los mismos parámetros que la LSTM con `H = 128`.

### El relleno importa (esto es lo que hay que cuidar)

Las instancias de un batch se rellenan con ceros hasta el largo de la más larga. En la LSTM unidireccional el relleno queda **al final**, después de las palabras reales, así que no las afecta. En la bidireccional, la pasada de derecha a izquierda **arranca en el relleno**: los ceros entran al estado oculto antes de llegar a la última palabra real y lo contaminan. El modelo correría sin errores pero rendiría peor.

La solución es `pack_padded_sequence`: le pasa a la LSTM el largo real de cada instancia y ninguna dirección procesa el relleno. Los largos se sacan del propio `x`: el relleno son filas de ceros exactos y ningún embedding real lo es. Así el modelo mantiene la misma firma `modelo(x)` que la LSTM y todo el código de `busqueda_lstm.py` funciona sin tocarlo.

In [ ]:
class PuntBiLSTM(nn.Module):
    def __init__(self, D=768, H=128, L=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(D, H, num_layers=L, batch_first=True, dropout=dropout, bidirectional=True)
        self.head_ini = nn.Linear(2 * H, 2)   # nada / ¿
        self.head_fin = nn.Linear(2 * H, 4)   # nada / , / . / ?
        self.head_cap = nn.Linear(2 * H, 4)   # 0 / 1 / 2 / 3

    def forward(self, x):
        T = x.shape[1]
        largos = (x != 0).any(dim=-1).sum(dim=1)   # palabras reales de cada instancia (el relleno es todo ceros)
        paquete = pack_padded_sequence(x, largos.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(paquete)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=T)   # (B, T, 2H); relleno = 0
        return self.head_ini(out), self.head_fin(out), self.head_cap(out)


PuntLSTM_uni = bl.PuntLSTM   # la unidireccional, solo para comparar parámetros
bl.PuntLSTM = PuntBiLSTM     # busqueda_lstm.entrenar() construye el modelo con este nombre

### Verificaciones

1. **Relleno:** la salida de una instancia tiene que ser la misma corriendo sola que dentro de un batch rellenado. Con `pack_padded_sequence` coincide; sin él, no (se muestra para ver el problema).
2. **Parámetros:** cuentas esperadas a mano. Capa 1: `2 · 4H(D + H + 2)`; capa 2: `2 · 4H(2H + H + 2)`; cabezas: `2H · (2 + 4 + 4) + 10`.

In [ ]:
def n_params(m):
    return sum(p.numel() for p in m.parameters())


# 1) relleno
torch.manual_seed(0)
m = PuntBiLSTM(H=32, L=2).eval()
corta, larga = torch.randn(1, 5, 768), torch.randn(1, 9, 768)
lote = torch.zeros(2, 9, 768)
lote[0, :5], lote[1] = corta[0], larga[0]                 # la instancia corta lleva 4 filas de relleno
with torch.no_grad():
    sola = m(corta)[1]                                    # logits de punt_final, sin relleno
    con_paquete = m(lote)[1][0:1, :5]                     # la misma instancia dentro del lote
    sin_paquete = m.head_fin(m.lstm(lote)[0])[0:1, :5]    # lo que pasaría sin pack_padded_sequence
print(f"diferencia máxima con paquete: {(sola - con_paquete).abs().max().item():.2e}   (tiene que ser ~0)")
print(f"diferencia máxima sin paquete: {(sola - sin_paquete).abs().max().item():.2e}   (el relleno contamina)")
assert (sola - con_paquete).abs().max().item() < 1e-5

# 2) parámetros
def esperados(H, L=2, D=768):
    capa1 = 2 * 4 * H * (D + H + 2)
    resto = sum(2 * 4 * H * (2 * H + H + 2) for _ in range(L - 1))
    return capa1 + resto + 2 * H * (2 + 4 + 4) + 10

for H in (64, 128):
    real = n_params(PuntBiLSTM(H=H, L=2))
    print(f"BiLSTM  L=2 H={H:3d}: {real:>9,} parámetros (esperado {esperados(H):,})")
    assert real == esperados(H)
print(f"LSTM uni L=2 H=128: {n_params(PuntLSTM_uni(H=128, L=2)):>9,} parámetros (referencia)")

---
# Paso 2: datos y configuración de la búsqueda

`bl.Datos` carga los embeddings y las etiquetas y arma los tres conjuntos exactamente igual que en la LSTM: test 20 % (`random_state=42`) y, del resto, validación 15 % (`random_state=0`), todo **por instancia**. Hay que mantenerlo así para que los números sean comparables con los de la LSTM.

> **Pendiente del equipo:** partir por película / fuente en vez de por instancia (hoy hay fuga: fragmentos de la misma película caen en train y en test, y el test sale optimista). Cuando se decida, hay que repetir la LSTM y la bidireccional con el mismo split nuevo; mientras tanto, ambas usan el actual.

**`RAPIDO = True`** es una prueba de humo: pocos datos, 2 épocas y grillas mínimas, para ver en un minuto que todo corre. Los resultados van a `salidas/bilstm_rapido/`, así que no se mezclan con los reales (`salidas/bilstm/`). Una vez que anda, se pone `RAPIDO = False`.

**Config de partida:** `H = 64` por dirección (comparable en parámetros con la LSTM de `H = 128`), `L = 2`, `dropout = 0,2`, `lr = 3e-3`, `potencia = 0,5`, alphas 1:1:1.

In [ ]:
RAPIDO = False   # True: prueba de humo. Cambiar a False para la búsqueda de verdad.

CARPETA_RES = bl.SALIDAS / ("bilstm_rapido" if RAPIDO else "bilstm")   # carpeta propia: el registro de corridas no distingue modelos
CARPETA_RES.mkdir(parents=True, exist_ok=True)

GRILLAS_BI = {
    "A": {"L": [1, 2], "H": [64, 128, 256]},   # H por dirección
    "B": {"dropout": [0.2, 0.4], "lr": [1e-3, 3e-3], "weight_decay": [0.0, 1e-5]},
    "C": {"potencia": [0.25, 0.5, 0.75]},
    "alphas": [],
    "alphas_cabeza": [0.5, 1.0, 2.0],   # peso de cada cabeza en la pérdida, una cabeza por vez (6 corridas)
}
GRILLAS_BI_RAPIDO = {
    "A": {"L": [1, 2], "H": [64]},
    "B": {"dropout": [0.2, 0.3], "lr": [1e-3, 3e-3], "weight_decay": [0.0]},
    "C": {"potencia": [0.25, 0.5]},
    "alphas": [],
}
CFG_BI = {**bl.CFG_DEFECTO, "H": 64, "lr": 3e-3}   # punto de partida de la búsqueda

datos = bl.Datos(rapido=RAPIDO)
ctx = argparse.Namespace(
    device=device, datos=datos, grillas=GRILLAS_BI_RAPIDO if RAPIDO else GRILLAS_BI,
    n_epocas=2 if RAPIDO else 12, n_epocas_final=2 if RAPIDO else 20, semillas_final=2 if RAPIDO else 3,
    margen=0.005, min_ganancia_umbral=0.005, buscar_alphas=not RAPIDO,
    csv_final=CARPETA_RES / "resultados_finales.csv", ruta_modelo=CARPETA_RES / "modelo_final.pt",
)
ctx.registro = bl.Registro(CARPETA_RES / "busqueda_script.csv", datos.huella)
RUTA_ESTADO = CARPETA_RES / "estado_busqueda.json"


def correr_etapa(nombre):
    """Corre una etapa de busqueda_lstm.py con el modelo bidireccional y guarda la config elegida."""
    todos = json.loads(RUTA_ESTADO.read_text(encoding="utf8")) if RUTA_ESTADO.exists() else {}
    estado = todos.get(datos.huella, {})
    cfg = {**CFG_BI, **estado.get("cfg", {})}
    t0 = time.time()
    print(f"=== etapa {nombre} | partida: {cfg}")
    cfg = bl.ETAPAS[nombre](ctx, cfg, estado)
    estado["cfg"] = cfg
    if nombre not in estado.setdefault("etapas_hechas", []):
        estado["etapas_hechas"].append(nombre)
    todos[datos.huella] = estado                       # se guarda tras cada etapa: se puede cortar y retomar
    RUTA_ESTADO.write_text(json.dumps(todos, indent=1, ensure_ascii=False), encoding="utf8")
    print(f"etapa {nombre} lista en {(time.time() - t0) / 60:.1f} min | config elegida: {cfg}")
    return cfg

---
# Paso 3: la búsqueda, etapa por etapa

Cada celda es una etapa; se pueden correr de a una. Cada etapa parte de la mejor config de la anterior. **Costo:** una época de la bidireccional es aproximadamente el doble de una de la LSTM (dos recorridos por capa), así que conviene correr en GPU (Colab) y empezar con `RAPIDO = True`.

### Etapa A: capas (`L`) y tamaño oculto por dirección (`H`)

In [ ]:
cfg = correr_etapa("A")

### Etapa B: dropout, learning rate y weight decay
Un eje a la vez, en ese orden. Con la bidireccional el modelo tiene más capacidad, así que el dropout más alto (0,4) puede ayudar.

In [ ]:
cfg = correr_etapa("B")

### Etapa C: `potencia` de los pesos de clase
Con `potencia = 1` la LSTM predecía las clases raras de más y empeoraba; por eso la grilla se queda entre 0,25 y 0,75.

Las **alphas** (peso de cada cabeza en la pérdida: `a_ini`, `a_fin`, `a_cap`) se buscan al final de esta etapa, una cabeza por vez con valores 0,5 / 1 / 2 (6 corridas más). En `RAPIDO` no se buscan.

In [ ]:
cfg = correr_etapa("C")

### Etapa D: umbral por clase
Reentrena la config elegida y mide si ajustar un factor por clase sobre las probabilidades mejora el F1 **en mitades de validación no vistas**. Solo se usa si la mejora media llega a 0,005.

In [ ]:
cfg = correr_etapa("D")

### Etapa final: 3 semillas y test
Reentrena la config elegida con 3 semillas (20 épocas, checkpoint por validación), ajusta el umbral con val si la etapa D lo aprobó y mide en **test**: es el único contacto con test. Guarda el mejor modelo según validación (no según test) y agrega las filas a `resultados_finales.csv`.

In [ ]:
cfg = correr_etapa("final")

---
# Paso 4: resultados y comparación con la LSTM

In [ ]:
res = pd.read_csv(CARPETA_RES / "busqueda_script.csv")
res = res[res.dataset == datos.huella]
cols = ["etapa", "L", "H", "dropout", "lr", "weight_decay", "potencia", "mejor_epoca", "score_val",
        "f1_punt_inicial", "f1_punt_final", "f1_capitalizacion", "segundos"]
print(f"{len(res)} corridas de búsqueda con el dataset {datos.huella}. Las 12 mejores por score de validación:\n")
print(res.sort_values("score_val", ascending=False)[cols].head(12).to_string(index=False))
print("\nRecordatorio: diferencias de score menores a ~0,015 no son concluyentes (las clases 2 y 3 de capitalización tienen muy pocas palabras en val).")

In [ ]:
def resumen_final(ruta, huella):
    if not Path(ruta).exists():
        return None
    d = pd.read_csv(ruta)
    d = d[d.dataset == huella]
    if d.empty:
        return None
    # la etapa final agrega filas cada vez que se corre: nos quedamos con la última config y una fila por semilla
    for c in ["H", "L", "dropout", "lr", "weight_decay", "potencia", "a_ini", "a_fin", "a_cap"]:
        d = d[np.isclose(d[c].astype(float), float(d.iloc[-1][c]))]
    d = d.drop_duplicates("seed", keep="last")
    cols = ["test_" + n for n in bl.NOMBRES] + ["score_test"]
    return pd.DataFrame({"media": d[cols].mean(), "desvío": d[cols].std(ddof=0), "semillas": len(d)}).round(3)


bi = resumen_final(CARPETA_RES / "resultados_finales.csv", datos.huella)
uni = resumen_final(bl.SALIDAS / "lstm" / "resultados_finales.csv", datos.huella)   # lo deja `python src/busqueda_lstm.py --etapas final`
print("BiLSTM en TEST (media y desvío entre semillas):")
print(bi.to_string() if bi is not None else "  todavía no hay: correr la etapa final")
print("\nLSTM unidireccional en TEST, mismo dataset:")
print(uni.to_string() if uni is not None else
      "  no hay resultados de la LSTM para este dataset: correr `python src/busqueda_lstm.py --etapas final` con los mismos .npy y .csv")
if bi is not None and uni is not None:
    print("\ndiferencia (BiLSTM - LSTM) en la media:")
    print((bi["media"] - uni["media"]).round(3).to_string())

In [ ]:
!python busqueda_lstm.py --etapas A B C D final --alphas

---
### Cómo leer la comparación
* Si la diferencia entre la media de la BiLSTM y la de la LSTM es **menor que el desvío entre semillas**, no se puede afirmar que una sea mejor.
* Lo esperable (hipótesis, no medido) es que la mejora aparezca en `punt_final` y en `punt_inicial` (`¿`), que dependen de lo que viene después, y no en `capitalizacion`.
* Para el póster: reportar ambas con el mismo split, los mismos datos y media ± desvío, junto con el número de parámetros de cada una.